# Estadisticas descriptivas de grupo (PD vs Control)
Notebook independiente: lee los CSV que ya genera `Script2.ipynb` (`ALL_patients_summary_pd.csv` y `ALL_patients_summary_control.csv`) y calcula media, desvio estandar y mediana **entre pacientes** para cada metrica de marcha, por grupo.

No corre nada de skdh ni toca las senales crudas -- solo trabaja sobre los resumenes ya generados.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import pandas as pd


## 2. Configuracion

In [ ]:
PD_SUMMARY_CSV      = "/content/drive/MyDrive/Lasige/PD/ALL_patients_summary_pd.csv"
CONTROL_SUMMARY_CSV = "/content/drive/MyDrive/Lasige/Control/ALL_patients_summary_control.csv"
BOUTS_VS_ACTIVITY_CSV = "/content/drive/MyDrive/Lasige/ALL_patients_bouts_vs_activity.csv"

# Pacientes a excluir del analisis (ej. fallos de deteccion conocidos: p624, p322)
EXCLUDED_PATIENTS = ["p624", "p322"]

OUTPUT_DIR = "/content/drive/MyDrive/Lasige"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "Group_descriptive_stats.csv")


Ajusta las rutas de arriba si tus CSV estan en otro sitio (son los que genera `Script2.ipynb` al final de `process_pd()` / `process_datos_control()`).

## 3. Cargar y combinar los resumenes de PD y Control

In [ ]:
pd_df = pd.read_csv(PD_SUMMARY_CSV)
control_df = pd.read_csv(CONTROL_SUMMARY_CSV)

combined = pd.concat([pd_df, control_df], ignore_index=True)
print(f"PD: {len(pd_df)} pacientes | Control: {len(control_df)} pacientes")

if EXCLUDED_PATIENTS:
    before = len(combined)
    found = combined.loc[combined["patient_id"].isin(EXCLUDED_PATIENTS), "patient_id"].tolist()
    not_found = [p for p in EXCLUDED_PATIENTS if p not in found]
    combined = combined[~combined["patient_id"].isin(EXCLUDED_PATIENTS)].reset_index(drop=True)
    print(f"Excluidos: {found} ({before} -> {len(combined)} pacientes)")
    if not_found:
        print(f"[AVISO] no se encontraron en los datos: {not_found}")

combined.head()


## 3b. Recalcular n_bouts y n_steps desde ALL_patients_bouts_vs_activity.csv

En vez de usar los `n_bouts`/`n_steps` que ya traia el resumen, los recalculamos directamente desde el fichero de bouts (contando solo los que tienen `matched_activity == "Walking slow"`), que es la fuente de datos original.

In [ ]:
bouts_df = pd.read_csv(BOUTS_VS_ACTIVITY_CSV)

walking_bouts = bouts_df[bouts_df["matched_activity"] == "Walking slow"]

bouts_summary = (
    walking_bouts
    .groupby("patient_id")
    .agg(
        n_bouts=("bout_n", "count"),
        n_steps=("n_steps", "sum"),
        total_duration_s=("duration_s", "sum"),
    )
    .reset_index()
)

print(f"Pacientes con al menos un bout de 'Walking slow': {len(bouts_summary)}")
bouts_summary.head()


In [ ]:
# quitamos los n_bouts/n_steps viejos del resumen y ponemos los recalculados
combined = combined.drop(columns=[c for c in ["n_bouts", "n_steps"] if c in combined.columns])
combined = combined.merge(bouts_summary, on="patient_id", how="left")

# pacientes sin ningun bout de Walking slow quedan en 0, no en NaN
combined[["n_bouts", "n_steps", "total_duration_s"]] = (
    combined[["n_bouts", "n_steps", "total_duration_s"]].fillna(0)
)

combined.head()


## 4. Tabla por paciente (para ver quien es quien)

Una fila por paciente, con `n_bouts`, `n_steps` y las metricas principales -- util para identificar outliers antes de mirar solo los promedios de grupo.

In [ ]:
# columnas clave a mostrar primero; ajusta esta lista si quieres ver otras metricas
key_metric_cols = [
    "PARAM:cadence_mean",
    "PARAM:gait speed_mean",
    "PARAM:stride time_mean",
]
key_metric_cols = [c for c in key_metric_cols if c in combined.columns]

per_patient_table = combined[
    ["patient_id", "Group", "n_bouts", "n_steps", "total_duration_s"] + key_metric_cols
].sort_values(["Group", "patient_id"]).reset_index(drop=True)

per_patient_path = os.path.join(OUTPUT_DIR, "Per_patient_table.csv")
per_patient_table.to_csv(per_patient_path, index=False)
print(f"Guardado -> {per_patient_path}")

per_patient_table


Si quieres ver **todas** las columnas por paciente (no solo las clave), usa `combined.sort_values(["Group", "patient_id"])` directamente.

## 5. Calcular estadisticas descriptivas por metrica y grupo

Incluye `n_bouts` y `n_steps` (conteos por paciente) ademas de todas las columnas `PARAM:*_mean`. Usamos solo las columnas `_mean` de cada paciente (la variabilidad **entre pacientes**), no las `_std` (que son variabilidad **dentro** de cada paciente, entre sus propios ciclos de marcha).

In [ ]:
# columnas de metricas de marcha por paciente (las que terminan en _mean)
param_metric_cols = [c for c in combined.columns if c.startswith("PARAM:") and c.endswith("_mean")]

# + los conteos por paciente, que no llevan prefijo PARAM:
count_cols = [c for c in ["n_bouts", "n_steps", "total_duration_s"] if c in combined.columns]

metric_cols = count_cols + param_metric_cols

print(f"{len(metric_cols)} metricas detectadas:")
for c in metric_cols:
    print(f"  {c}")


In [ ]:
rows = []

for col in metric_cols:
    metric_name = col.replace("PARAM:", "").replace("_mean", "")
    for group_name, group_df in combined.groupby("Group"):
        values = group_df[col].dropna()
        rows.append({
            "metric": metric_name,
            "Group": group_name,
            "n": len(values),
            "mean": values.mean(),
            "std": values.std(),
            "median": values.median(),
            "min": values.min(),
            "max": values.max(),
        })

group_stats = pd.DataFrame(rows)
group_stats = group_stats.sort_values(["metric", "Group"]).reset_index(drop=True)
group_stats


## 6. Guardar el resultado

In [ ]:
group_stats.to_csv(OUTPUT_CSV, index=False)
print(f"Guardado -> {OUTPUT_CSV}")


## 7. (Opcional) Ver una metrica especifica lado a lado

Ejemplo rapido para revisar una metrica en concreto sin tener que buscarla en la tabla larga.

In [ ]:
METRIC_TO_CHECK = "cadence"  # cambia por cualquier nombre de la lista de la Celda 4

group_stats[group_stats["metric"] == METRIC_TO_CHECK]


## 8. Graficos para el reporte (boxplots PD vs Control)

Boxplot en vez de barras con error, porque ya vimos que hay outliers (p322, p624...) que una media +/- std esconde. El boxplot muestra la mediana, el rango intercuartil, y marca los outliers como puntos aparte -- mas honesto para un reporte con muestras pequenas.

Ajusta `REPORT_METRICS` con las metricas que quieras incluir (usa los nombres de columna reales, los mismos que aparecen en la Celda 4/12).

In [ ]:
import matplotlib.pyplot as plt

REPORT_METRICS = {
    "PARAM:cadence_mean": "Cadence (steps/min)",
    "PARAM:gait speed_mean": "Gait speed (m/s)",
    "PARAM:stride time_mean": "Stride time (s)",
    "PARAM:step length_mean": "Step length (m)",
    "PARAM:harmonic ratio - V_mean": "Harmonic ratio (V)",
    "PARAM:stride time asymmetry_mean": "Stride time asymmetry",
}
report_cols = {k: v for k, v in REPORT_METRICS.items() if k in combined.columns}
missing = [k for k in REPORT_METRICS if k not in combined.columns]
if missing:
    print(f"[WARNING] these columns don't exist in your data, skipping: {missing}")


In [ ]:
groups = sorted(combined["Group"].unique())
n = len(report_cols)
n_cols = 3
n_rows = -(-n // n_cols)  # techo

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
axes = axes.flatten() if n > 1 else [axes]

for ax, (col, label) in zip(axes, report_cols.items()):
    data = [combined.loc[combined["Group"] == g, col].dropna() for g in groups]
    ax.boxplot(data, showmeans=True)
    ax.set_xticks(range(1, len(groups) + 1))
    ax.set_xticklabels(groups)
    ax.set_title(label)
    ax.set_ylabel(label)

# apaga los paneles sobrantes si las metricas no llenan la grilla
for ax in axes[n:]:
    ax.axis("off")

plt.tight_layout()
boxplot_path = os.path.join(OUTPUT_DIR, "Report_boxplots.png")
plt.savefig(boxplot_path, dpi=200)
print(f"Guardado -> {boxplot_path}")
plt.show()


## 9. Tabla para el reporte (media +/- std, mediana)

Formato "mean +/- std (median)" por metrica y grupo, en tabla ancha (una fila por metrica, una columna por grupo) -- lista para copiar a Word/Excel o exportar como imagen.

In [ ]:
def format_cell(row):
    return f"{row['mean']:.2f} \u00b1 {row['std']:.2f} ({row['median']:.2f})"

report_table = group_stats.copy()
report_table["formatted"] = report_table.apply(format_cell, axis=1)

report_wide = report_table.pivot(index="metric", columns="Group", values="formatted")
n_per_group = combined.groupby("Group").size()
report_wide.columns = [f"{g} (n={n_per_group[g]})" for g in report_wide.columns]

report_table_path = os.path.join(OUTPUT_DIR, "Report_table.csv")
report_wide.to_csv(report_table_path)
print(f"Guardado -> {report_table_path}")

report_wide


## 10. (Opcional) Exportar la tabla como imagen

Util si tu reporte va en un formato donde prefieres pegar una imagen en vez de importar un CSV (ej. una presentacion, o un documento sin edicion facil de tablas).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 0.4 * len(report_wide) + 1))
ax.axis("off")

tbl = ax.table(
    cellText=report_wide.values,
    rowLabels=report_wide.index,
    colLabels=report_wide.columns,
    cellLoc="center",
    loc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.5)

plt.tight_layout()
table_img_path = os.path.join(OUTPUT_DIR, "Report_table.png")
plt.savefig(table_img_path, dpi=200, bbox_inches="tight")
print(f"Guardado -> {table_img_path}")
plt.show()


## 11. Correlation heatmap

Correlacion (Pearson) entre las metricas clave, usando todos los pacientes (PD + Control juntos). Util para ver, por ejemplo, si `gait speed` y `cadence` estan fuertemente correlacionadas (esperable), o si alguna metrica de asimetria se relaciona con la velocidad de marcha.

Si prefieres ver la correlacion SOLO dentro de un grupo (ej. solo PD), cambia `CORR_GROUP` de `None` a `"PD"` o `"Control"`.

In [ ]:
import numpy as np

CORR_GROUP = None  # None = todos los pacientes juntos; o "PD" / "Control"

corr_source = combined if CORR_GROUP is None else combined[combined["Group"] == CORR_GROUP]
corr_cols = list(report_cols.keys())  # reusa las mismas metricas de los boxplots
corr_matrix = corr_source[corr_cols].corr(method="pearson")
corr_labels = [report_cols[c] for c in corr_cols]

fig, ax = plt.subplots(figsize=(1.1 * len(corr_cols) + 2, 1.1 * len(corr_cols) + 2))
im = ax.imshow(corr_matrix.values, vmin=-1, vmax=1, cmap="RdBu_r")

ax.set_xticks(range(len(corr_labels)))
ax.set_yticks(range(len(corr_labels)))
ax.set_xticklabels(corr_labels, rotation=45, ha="right")
ax.set_yticklabels(corr_labels)

for i in range(len(corr_labels)):
    for j in range(len(corr_labels)):
        ax.text(j, i, f"{corr_matrix.values[i, j]:.2f}",
                ha="center", va="center", fontsize=8,
                color="white" if abs(corr_matrix.values[i, j]) > 0.6 else "black")

title_suffix = "all patients" if CORR_GROUP is None else CORR_GROUP
ax.set_title(f"Correlation matrix ({title_suffix})")
fig.colorbar(im, ax=ax, label="Pearson r")

plt.tight_layout()
corr_path = os.path.join(OUTPUT_DIR, "Report_correlation_heatmap.png")
plt.savefig(corr_path, dpi=200, bbox_inches="tight")
print(f"Guardado -> {corr_path}")
plt.show()


## 12. (Opcional) Scatter plot de un par especifico

Si en el heatmap ves una correlacion que te interesa mirar en detalle (ej. gait speed vs. cadence), esta celda la dibuja como scatter, coloreado por grupo.

In [ ]:
X_METRIC = "PARAM:cadence_mean"
Y_METRIC = "PARAM:gait speed_mean"

fig, ax = plt.subplots(figsize=(6, 5))
for g in groups:
    sub = combined[combined["Group"] == g]
    ax.scatter(sub[X_METRIC], sub[Y_METRIC], label=g, alpha=0.8)

r = combined[[X_METRIC, Y_METRIC]].corr().iloc[0, 1]
ax.set_xlabel(REPORT_METRICS.get(X_METRIC, X_METRIC))
ax.set_ylabel(REPORT_METRICS.get(Y_METRIC, Y_METRIC))
ax.set_title(f"r = {r:.2f}")
ax.legend()

plt.tight_layout()
scatter_path = os.path.join(OUTPUT_DIR, "Report_scatter.png")
plt.savefig(scatter_path, dpi=200, bbox_inches="tight")
print(f"Guardado -> {scatter_path}")
plt.show()
